# Comprehensive 2D Prediction Run Analysis (Flow Matching)

This notebook provides a complete analysis of a 2D prediction run from the **Flow Matching** model. It loads the `prediction_stats.npz` and timeseries files generated by `gen_prediction_fm.py` to visualize:

1.  **Quantitative Metrics:** Overall and per-channel R² scores.
2.  **Error Plots:** R² bar charts and error-over-time plots.
3.  **Qualitative Visualizations:** Direct visual comparisons of ground truth, predictions, and errors, both as spatial snapshots and as time-evolution plots.

### 1. Setup and Imports

In [ ]:
from pathlib import Path
import numpy as np
import logging
import matplotlib.pyplot as plt

# Ensure the mhd_surrogate_core package is in your PYTHONPATH
from mhd_surrogate_core.plotting.xz import (
    plot_prediction_rollout_error,
    plot_r2_performance,
    plot_xz_snapshot,
    plot_x_time_evolution,
    plot_z_time_evolution,
    plot_velocity_quiver
)

### 2. Master Configuration

**This is the only cell you need to edit.**

Set all paths, plotting preferences, and slice indices here. Then, you can "Run All" cells.

In [ ]:
# === 1. CONFIG FILE PATH ===
# This points to the config file read by the bash script
CONFIG_FILE = Path("evaluation.conf")

# === 2. CHANNEL NAME MAPPING ===
channel_map = {
    'vx': 'u',
    'vz': 'w'
}

# === 3. GLOBAL PLOT SIZING ===
global_base_size = 15
global_min_size = 3

# === 4. UNIT LABEL AND COLORMAPS ===
global_unit_label = ""
main_cmap = "seismic"      # For ground truth and prediction
diff_cmap = "seismic"      # For difference plots (blue-white-red is good for errors)
magnitude_cmap = 'inferno' # For magnitude quiver plots
diff_magnitude_cmap = 'Reds' # For quiver plot of the difference field

# === 5. COLOR SCALES (MAIN PLOTS) ===
# Tuned for Re16k data range. Set to None to auto-scale.
main_vmins = {
    'vx': -2.2,
    'vz': -3.0,
}
main_vmaxs = {
    'vx': 3.8,
    'vz': 3.0,
}
main_vcenters = {
    'vx': 0.84,
    'vz': 0.0
}

# === 6. COLOR SCALES (DIFFERENCE PLOTS) ===
diff_vmins = {
    'vx': -1.0,
    'vz': -1.0,
}
diff_vmaxs = {
    'vx': 1.0,
    'vz': 1.0,
}
diff_vcenters = {
    'vx': 0.0,
    'vz': 0.0
}

# === 7. QUIVER PLOT MEAN FLOW ===
global_mean_flow = {
    'vx': 0.84
}

# === 8. SPATIAL SNAPSHOT PARAMETERS (Section 5) ===
plot_time_index = 8
quiver_downsample_stride = 15
quiver_arrow_width = 0.0025

# === 9. TIME EVOLUTION PARAMETERS (Section 6) ===
plot_x_index_tz = 1000 # X-index for Time-Z plot
plot_z_index_tx = 8  # Z-index for Time-X plot

print("Master configuration set.")

### 3. Load Paths and Data

**This cell contains all loading logic.**

It reads the `CONFIG_FILE` set above, finds the data files, and loads them into memory. You should not need to edit this cell.

In [ ]:
# --- 1. CONFIG PARSING FUNCTION ---
def parse_bash_config(config_file):
    """A simple parser for the .conf file.
    
    Handles 'KEY="value"' or 'KEY=value' and ignores comments.
    """
    config = {}
    if not config_file.exists():
        raise FileNotFoundError(f"Configuration file not found: {config_file}\n" 
                                "Please run 'run_full_evaluation.sh' or create 'evaluation.conf'.")
    with open(config_file, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
          
            if '=' in line:
                key, value = line.split('=', 1)
                key = key.strip()
                # Strip quotes (handles both ' and "")
                value = value.strip().strip('"').strip("'")
                config[key] = value
    return config

# --- 2. PATH LOADING --- 
try:
    parsed_config = parse_bash_config(CONFIG_FILE)
  
    if 'MODEL_PATH' not in parsed_config or 'TEST_DATA_PATH' not in parsed_config:
        raise KeyError("Config file must contain MODEL_PATH and TEST_DATA_PATH")
    
    # Derive the eval output dir from the MODEL_PATH, just like the bash script does
    model_path = Path(parsed_config['MODEL_PATH'])
    eval_output_dir = model_path.parent / "eval" / "pred"
    
    # Get the ground truth path directly
    ground_truth_path = Path(parsed_config['TEST_DATA_PATH'])
    
    print(f"--- Auto-loaded paths from {CONFIG_FILE} ---")
    print(f"Using Eval Dir: {eval_output_dir}")
    print(f"Using GT Path:  {ground_truth_path}")

except Exception as e:
    print(f"FATAL: Could not load configuration. Error: {e}")
    print("--- Using fallback paths. PLEASE CHECK YOUR 'evaluation.conf' FILE! ---")
    # Fallback to the original hardcoded paths
    eval_output_dir = Path("/cephfs/users/skowronek/Documents/PhD/nuclear_fusion_cooling/prediction/mhd_surrogate_modelling/experiments/study_6_flow_matching_2d/study_6.1_flow_matching_2d_base/output/sweep_valrol_steps8/f64-128-256_t64_vroll8/eval/pred/")
    ground_truth_path = Path("/cephfs/users/skowronek/Documents/PhD/nuclear_fusion_cooling/data/preprocessed_dns_output/01-Cold_Runs/01-Re16K_Ha325/T1492_x1151_y1_z127_c2/preprocessed/test_set.npz")

# --- 3. DATA LOADING LOGIC ---
stats_file_path = eval_output_dir / "prediction_stats.npz"
predicted_file_path = eval_output_dir / "predicted_timeseries.npz"
difference_file_path = eval_output_dir / "difference_timeseries.npz"

if not all([p.exists() for p in [stats_file_path, predicted_file_path, difference_file_path, ground_truth_path]]):
    logging.error("ERROR: One or more data files not found. Please check the paths.")
    print(f"Missing: {stats_file_path}" if not stats_file_path.exists() else "")
    print(f"Missing: {predicted_file_path}" if not predicted_file_path.exists() else "")
    print(f"Missing: {difference_file_path}" if not difference_file_path.exists() else "")
    print(f"Missing: {ground_truth_path}" if not ground_truth_path.exists() else "")
else:
    with np.load(stats_file_path, allow_pickle=True) as data:
        r_squared_total = data['r_squared_total']
        channel_names = list(data['channel_names'])
        # Try loading extra FM metadata if it exists
        if 'ode_settings' in data:
             print(f"ODE Settings: Solver={data['ode_settings'][0]}, Steps={data['ode_settings'][1]}")
        print(f"Loaded statistics from {stats_file_path}")
    
    with np.load(predicted_file_path, allow_pickle=True) as data:
        predicted_timeseries = data['timeseries']
        print(f"Loaded predicted timeseries from {predicted_file_path}")

    with np.load(difference_file_path, allow_pickle=True) as data:
        difference_timeseries = data['timeseries']
        print(f"Loaded difference timeseries from {difference_file_path}")
        
    with np.load(ground_truth_path, allow_pickle=True) as data:
        ground_truth_timeseries_all = data['timeseries']
        all_labels = list(data['labels'])
        if ground_truth_timeseries_all.ndim == 5:
            ground_truth_timeseries_all = np.squeeze(ground_truth_timeseries_all, axis=2)
        coords = {
            'x': data.get('x_coords', np.arange(ground_truth_timeseries_all.shape[1])),
            'z': data.get('z_coords', np.arange(ground_truth_timeseries_all.shape[2])),
            'labels': all_labels # Use all labels for indexing
        }
        print(f"Loaded ground truth from {ground_truth_path}")
    
    # Filter ground truth to only include channels the model was trained on
    gt_indices = [all_labels.index(name) for name in channel_names]
    ground_truth_timeseries = ground_truth_timeseries_all[..., gt_indices]
    # IMPORTANT: Update coords to match the filtered data
    coords['labels'] = channel_names
    print(f"Filtered ground truth to match predicted channels: {channel_names}")
    
    # Check that shapes match
    if ground_truth_timeseries.shape[0] < predicted_timeseries.shape[0]:
        print(f"Warning: Ground truth ({ground_truth_timeseries.shape[0]}) is shorter than prediction ({predicted_timeseries.shape[0]}). Truncating prediction.")
        T = ground_truth_timeseries.shape[0]
        predicted_timeseries = predicted_timeseries[:T]
        difference_timeseries = difference_timeseries[:T]
    elif predicted_timeseries.shape[0] < ground_truth_timeseries.shape[0]:
        print(f"Warning: Prediction ({predicted_timeseries.shape[0]}) is shorter than ground truth ({ground_truth_timeseries.shape[0]}). Truncating ground truth.")
        T = predicted_timeseries.shape[0]
        ground_truth_timeseries = ground_truth_timeseries[:T]
        difference_timeseries = difference_timeseries[:T]
    
    if ground_truth_timeseries.shape != predicted_timeseries.shape:
        logging.error(f"FATAL: Shapes still do not match! GT: {ground_truth_timeseries.shape}, Pred: {predicted_timeseries.shape}")
    else:
        print(f"Data loaded and aligned. Final shape: {ground_truth_timeseries.shape}")

    # --- 4. SET MAX INDICES (for validation) ---
    max_time_index = predicted_timeseries.shape[0] - 1
    max_x_index = predicted_timeseries.shape[1] - 1
    max_z_index = predicted_timeseries.shape[2] - 1
    print(f"\nMax indices -> Time: {max_time_index}, X: {max_x_index}, Z: {max_z_index}")
    
    velocity_components = sorted([c for c in coords.get('labels', []) if c in channel_map])
    print(f"Identified velocity components to plot: {velocity_components}")

---

### 4. Quantitative Analysis

In [ ]:
print(f"--- PREDICTION SUMMARY ---")
print(f"Overall R² Score: {r_squared_total:.4f}\n")

print("\n--- Average Prediction Performance (R²) ---")
plot_r2_performance(stats_file_path, eval_type="Prediction")

print("\n--- Prediction Error Over Time ---")
plot_prediction_rollout_error(stats_file_path)

---

### 5. Qualitative Analysis: Spatial Snapshots

This section compares the ground truth, prediction, and error for a single snapshot in time.
*(Parameters are set in the 'Master Configuration' cell at the top)*

In [ ]:
# === Validate Snapshot Parameters ===
if plot_time_index > max_time_index:
    plot_time_index = max_time_index
    print(f"Warning: plot_time_index too large. Clamping to {max_time_index}.")
print(f"Using time-index {plot_time_index} for snapshot plots.")

#### 5.1 Velocity Field Quiver Plot Comparison

In [ ]:
# === Plot Quiver Comparison ===
if 'vx' in velocity_components and 'vz' in velocity_components:
    # --- 1. Calculate shared color scale for truth and prediction fluctuations ---
    vx_idx = coords['labels'].index('vx')
    vz_idx = coords['labels'].index('vz')
    mean_u = global_mean_flow.get('vx', 0.0)
    mean_v = global_mean_flow.get('vz', 0.0)

    # Helper to calculate fluctuation magnitude
    def get_fluctuation_magnitude(timeseries):
        u_fluc = timeseries[plot_time_index, :, :, vx_idx].copy()
        v_fluc = timeseries[plot_time_index, :, :, vz_idx].copy()
        if u_fluc.shape[0] > 2 and u_fluc.shape[1] > 2: u_fluc[1:-1, 1:-1] -= mean_u
        if v_fluc.shape[0] > 2 and v_fluc.shape[1] > 2: v_fluc[1:-1, 1:-1] -= mean_v
        return np.sqrt(u_fluc**2 + v_fluc**2)
    
    gt_mag = get_fluctuation_magnitude(ground_truth_timeseries)
    pred_mag = get_fluctuation_magnitude(predicted_timeseries)
    
    quiver_vmin = min(gt_mag.min(), pred_mag.min())
    quiver_vmax = max(gt_mag.max(), pred_mag.max())
    print(f"Shared quiver color scale (fluctuations): [{quiver_vmin:.3f}, {quiver_vmax:.3f}]")

    # --- 2. Plot Ground Truth Quiver ---
    plot_velocity_quiver(
        data_path=None, timeseries_data=ground_truth_timeseries, coords=coords,
        time_index=plot_time_index, u_channel='vx', v_channel='vz',
        channel_alias_map=channel_map, mean_flow_components=globals().get('global_mean_flow'),
        vmin=quiver_vmin, vmax=quiver_vmax, base_size=global_base_size, min_size=global_min_size,
        unit_label=global_unit_label, cmap=magnitude_cmap, quiver_stride=quiver_downsample_stride,
        arrow_width=quiver_arrow_width, title_suffix="Ground Truth"
    )

    # --- 3. Plot Prediction Quiver ---
    plot_velocity_quiver(
        data_path=None, timeseries_data=predicted_timeseries, coords=coords,
        time_index=plot_time_index, u_channel='vx', v_channel='vz',
        channel_alias_map=channel_map, mean_flow_components=globals().get('global_mean_flow'),
        vmin=quiver_vmin, vmax=quiver_vmax, base_size=global_base_size, min_size=global_min_size,
        unit_label=global_unit_label, cmap=magnitude_cmap, quiver_stride=quiver_downsample_stride,
        arrow_width=quiver_arrow_width, title_suffix="Prediction"
    )

    # --- 4. Plot Difference Quiver ---
    cbar_label = "Magnitude of Difference" + (f" [{global_unit_label}]" if global_unit_label else "")
    plot_velocity_quiver(
        data_path=None, timeseries_data=difference_timeseries, coords=coords,
        time_index=plot_time_index, u_channel='vx', v_channel='vz',
        channel_alias_map=channel_map, mean_flow_components=None, # Plot raw difference
        base_size=global_base_size, min_size=global_min_size, unit_label=global_unit_label,
        cmap=diff_magnitude_cmap, quiver_stride=quiver_downsample_stride,
        arrow_width=quiver_arrow_width, title_suffix="Difference",
        cbar_label_override=cbar_label
    )

#### 5.2 Individual Component Snapshots

In [ ]:
# === Plot Snapshot Comparison for u (vx) ===
if 'vx' in velocity_components:
    # Calculate shared color scale for truth and prediction, unless overridden
    vmin = main_vmins.get('vx')
    vmax = main_vmaxs.get('vx')
    if vmin is None or vmax is None:
        vx_idx = coords['labels'].index('vx')
        gt_slice = ground_truth_timeseries[plot_time_index, :, :, vx_idx]
        pred_slice = predicted_timeseries[plot_time_index, :, :, vx_idx]
        auto_vmin = min(gt_slice.min(), pred_slice.min())
        auto_vmax = max(gt_slice.max(), pred_slice.max())
        if vmin is None: vmin = auto_vmin
        if vmax is None: vmax = auto_vmax

    # Ground Truth
    plot_xz_snapshot(data_path=None, timeseries_data=ground_truth_timeseries, coords=coords, channel='vx',
                     time_index=plot_time_index, channel_alias=f"{channel_map.get('vx')} (Ground Truth)",
                     vmin=vmin, vmax=vmax, vcenter=main_vcenters.get('vx'), base_size=global_base_size,
                     min_size=global_min_size, unit_label=global_unit_label, cmap=main_cmap)
    # Prediction
    plot_xz_snapshot(data_path=None, timeseries_data=predicted_timeseries, coords=coords, channel='vx',
                     time_index=plot_time_index, channel_alias=f"{channel_map.get('vx')} (Prediction)",
                     vmin=vmin, vmax=vmax, vcenter=main_vcenters.get('vx'), base_size=global_base_size, 
                     min_size=global_min_size, unit_label=global_unit_label, cmap=main_cmap)
    # Difference
    plot_xz_snapshot(data_path=None, timeseries_data=difference_timeseries, coords=coords, channel='vx',
                     time_index=plot_time_index, channel_alias=f"{channel_map.get('vx')} (Difference)",
                     vmin=diff_vmins.get('vx'), vmax=diff_vmaxs.get('vx'), vcenter=diff_vcenters.get('vx'), 
                     base_size=global_base_size, min_size=global_min_size, unit_label=global_unit_label, cmap=diff_cmap)

In [ ]:
# === Plot Snapshot Comparison for w (vz) ===
if 'vz' in velocity_components:
    vmin = main_vmins.get('vz')
    vmax = main_vmaxs.get('vz')
    if vmin is None or vmax is None:
        vz_idx = coords['labels'].index('vz')
        gt_slice = ground_truth_timeseries[plot_time_index, :, :, vz_idx]
        pred_slice = predicted_timeseries[plot_time_index, :, :, vz_idx]
        auto_vmin = min(gt_slice.min(), pred_slice.min())
        auto_vmax = max(gt_slice.max(), pred_slice.max())
        if vmin is None: vmin = auto_vmin
        if vmax is None: vmax = auto_vmax

    plot_xz_snapshot(data_path=None, timeseries_data=ground_truth_timeseries, coords=coords, channel='vz',
                     time_index=plot_time_index, channel_alias=f"{channel_map.get('vz')} (Ground Truth)",
                     vmin=vmin, vmax=vmax, vcenter=main_vcenters.get('vz'), base_size=global_base_size,
                     min_size=global_min_size, unit_label=global_unit_label, cmap=main_cmap)
    plot_xz_snapshot(data_path=None, timeseries_data=predicted_timeseries, coords=coords, channel='vz',
                     time_index=plot_time_index, channel_alias=f"{channel_map.get('vz')} (Prediction)",
                     vmin=vmin, vmax=vmax, vcenter=main_vcenters.get('vz'), base_size=global_base_size, 
                     min_size=global_min_size, unit_label=global_unit_label, cmap=main_cmap)
    plot_xz_snapshot(data_path=None, timeseries_data=difference_timeseries, coords=coords, channel='vz',
                     time_index=plot_time_index, channel_alias=f"{channel_map.get('vz')} (Difference)",
                     vmin=diff_vmins.get('vz'), vmax=diff_vmaxs.get('vz'), vcenter=diff_vcenters.get('vz'), 
                     base_size=global_base_size, min_size=global_min_size, unit_label=global_unit_label, cmap=diff_cmap)

---

### 6. Qualitative Analysis: Time Evolution

*(Parameters are set in the 'Master Configuration' cell at the top)*

#### 6.1 Evolution along Z-axis (at constant X)

In [ ]:
# === Validate Time-Z Parameters ===
if plot_x_index_tz > max_x_index:
    plot_x_index_tz = max_x_index
    print(f"Warning: plot_x_index_tz too large. Clamping to {max_x_index}.")
print(f"Using x-index {plot_x_index_tz} for Time-Z plots.")

In [ ]:
# === Plot Time-Z Evolution for u (vx) ===
if 'vx' in velocity_components:
    vmin = main_vmins.get('vx')
    vmax = main_vmaxs.get('vx')
    if vmin is None or vmax is None:
        vx_idx = coords['labels'].index('vx')
        gt_slice = ground_truth_timeseries[:, plot_x_index_tz, :, vx_idx]
        pred_slice = predicted_timeseries[:, plot_x_index_tz, :, vx_idx]
        auto_vmin = min(gt_slice.min(), pred_slice.min())
        auto_vmax = max(gt_slice.max(), pred_slice.max())
        if vmin is None: vmin = auto_vmin
        if vmax is None: vmax = auto_vmax

    plot_z_time_evolution(data_path=None, timeseries_data=ground_truth_timeseries, coords=coords, channel='vx',
                          x_index=plot_x_index_tz, channel_alias=f"{channel_map.get('vx')} (Ground Truth)",
                          vmin=vmin, vmax=vmax, vcenter=main_vcenters.get('vx'), base_size=global_base_size,
                          min_size=global_min_size, unit_label=global_unit_label, cmap=main_cmap)
    plot_z_time_evolution(data_path=None, timeseries_data=predicted_timeseries, coords=coords, channel='vx',
                          x_index=plot_x_index_tz, channel_alias=f"{channel_map.get('vx')} (Prediction)",
                          vmin=vmin, vmax=vmax, vcenter=main_vcenters.get('vx'), base_size=global_base_size,
                          min_size=global_min_size, unit_label=global_unit_label, cmap=main_cmap)
    plot_z_time_evolution(data_path=None, timeseries_data=difference_timeseries, coords=coords, channel='vx',
                          x_index=plot_x_index_tz, channel_alias=f"{channel_map.get('vx')} (Difference)",
                          vmin=diff_vmins.get('vx'), vmax=diff_vmaxs.get('vx'), vcenter=diff_vcenters.get('vx'),
                          base_size=global_base_size, min_size=global_min_size, unit_label=global_unit_label, cmap=diff_cmap)

#### 6.2 Evolution along X-axis (at constant Z)

In [ ]:
# === Validate Time-X Parameters ===
if plot_z_index_tx > max_z_index:
    plot_z_index_tx = max_z_index
    print(f"Warning: plot_z_index_tx too large. Clamping to {max_z_index}.")
print(f"Using z-index {plot_z_index_tx} for Time-X plots.")

In [ ]:
# === Plot Time-X Evolution for u (vx) ===
if 'vx' in velocity_components:
    vmin = main_vmins.get('vx')
    vmax = main_vmaxs.get('vx')
    if vmin is None or vmax is None:
        vx_idx = coords['labels'].index('vx')
        gt_slice = ground_truth_timeseries[:, :, plot_z_index_tx, vx_idx]
        pred_slice = predicted_timeseries[:, :, plot_z_index_tx, vx_idx]
        auto_vmin = min(gt_slice.min(), pred_slice.min())
        auto_vmax = max(gt_slice.max(), pred_slice.max())
        if vmin is None: vmin = auto_vmin
        if vmax is None: vmax = auto_vmax

    plot_x_time_evolution(data_path=None, timeseries_data=ground_truth_timeseries, coords=coords, channel='vx',
                          z_index=plot_z_index_tx, channel_alias=f"{channel_map.get('vx')} (Ground Truth)",
                          vmin=vmin, vmax=vmax, vcenter=main_vcenters.get('vx'), base_size=global_base_size,
                          min_size=global_min_size, unit_label=global_unit_label, cmap=main_cmap)
    plot_x_time_evolution(data_path=None, timeseries_data=predicted_timeseries, coords=coords, channel='vx',
                          z_index=plot_z_index_tx, channel_alias=f"{channel_map.get('vx')} (Prediction)",
                          vmin=vmin, vmax=vmax, vcenter=main_vcenters.get('vx'), base_size=global_base_size,
                          min_size=global_min_size, unit_label=global_unit_label, cmap=main_cmap)
    plot_x_time_evolution(data_path=None, timeseries_data=difference_timeseries, coords=coords, channel='vx',
                          z_index=plot_z_index_tx, channel_alias=f"{channel_map.get('vx')} (Difference)",
                          vmin=diff_vmins.get('vx'), vmax=diff_vmaxs.get('vx'), vcenter=diff_vcenters.get('vx'),
                          base_size=global_base_size, min_size=global_min_size, unit_label=global_unit_label, cmap=diff_cmap)